# Modelling

First model to predict the number of rings of an abalone. The runs are tracked with MLflow (`mlflow ui` to compare them).

In [1]:
import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/abalone.csv")
df = pd.get_dummies(df, columns=["Sex"], dtype=int)

X = df.drop(columns="Rings")
y = df["Rings"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
mlflow.set_experiment("abalone")

with mlflow.start_run(run_name="linear_regression"):
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.sklearn.log_model(model, name="model")

print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

2026/10/22 23:00:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RMSE: 2.21, MAE: 1.59, R2: 0.55


In [4]:
params = {"n_estimators": 200, "max_depth": 10}

with mlflow.start_run(run_name="random_forest"):
    model = RandomForestRegressor(**params, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_params(params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.sklearn.log_model(
        model, name="model", skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

2026/10/22 23:00:44 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RMSE: 2.23, MAE: 1.56, R2: 0.54


Both models are close: RMSE around 2.2 rings and R2 around 0.55 on the test set. The random forest is not better than the linear regression, so I keep the linear regression for the next steps.